In [ ]:
1. HR (Heart Rate) — 심박수

단위: BPM (Beats Per Minute)
측정 원리: BVP 신호의 주파수 도메인에서 피크 주파수를 BPM으로 변환
정상 범위: 안정 시 60~100 BPM
임상 의미:
< 60 BPM → 서맥 (Bradycardia)
> 100 BPM → 빈맥 (Tachycardia)



2. SQI (Signal Quality Index) — 신호 품질 지수

범위: 0.0 (노이즈) ~ 1.0 (완벽한 신호)
측정 원리: BVP 신호의 자기상관(Autocorrelation)을 이용해 주기성을 평가
중요성: 다른 모든 지표의 신뢰도를 결정하는 메타 지표
실용적 기준:

≥ 0.7 → 신뢰 가능
0.4~0.7 → 주의 필요
< 0.4 → 해당 윈도우 데이터 사용 지양

⚠️ SQI가 낮을 때의 HRV 값은 생리학적 의미가 없고, 오히려 노이즈입니다.




3. SDNN (Standard Deviation of NN intervals)

단위: ms
측정 원리: RR interval(연속 심박 간격)의 표준편차
정상 범위: 40~100 ms (안정 시 성인)
임상 의미: 전체 자율신경계 활성도를 반영

높을수록 → 건강한 자율신경 균형
낮을수록 → 심부전, 당뇨병성 신경병증, 심근경색 후 위험 상승
< 50 ms → 임상적 주의 기준 (일부 가이드라인)




4. RMSSD (Root Mean Square of Successive Differences)

단위: ms
측정 원리: 연속된 RR interval 차이의 제곱평균제곱근
정상 범위: 20~60 ms
임상 의미: 부교감신경(미주신경) 활성도에 가장 민감한 지표

스트레스/불안 → RMSSD 감소
회복/이완 → RMSSD 증가





5. pNN50

단위: % (퍼센트)
측정 원리: 연속 RR interval 차이가 50ms를 초과하는 쌍의 비율
정상 범위: 10~40%
임상 의미: RMSSD와 유사하게 부교감신경 활성도 반영

고령자, 심혈관 질환자 → 현저히 낮아짐
RMSSD와 높은 상관관계 (사실상 중복 정보가 많음)




6. LF/HF Ratio (Low Frequency / High Frequency)

단위: 무차원 비율
원리: BVP 신호의 주파수 스펙트럼 분석

LF (0.04~0.15 Hz): 교감신경 + 부교감신경 혼합
HF (0.15~0.4 Hz): 부교감신경(호흡과 연동)

정상 범위: 안정 시 1~2, 운동/스트레스 시 급등
임상 의미: 교감/부교감 균형 (자율신경 균형 지표)

높을수록 → 교감신경 우세 (스트레스, 통증, 심부전)
낮을수록 → 부교감 우세 (이완, 수면)
⚠️ 단독으로 사용하면 해석이 모호한 편 (학계에서 논쟁 중)






# ✅ 모델에 넣을 컬럼
AI_FEATURES = [
    # 1. 현재 실시간 절대값 (_raw)
    #    → "이 사람의 HR이 임상적으로 위험한 수치인가"
    'hr_raw', 'sdnn_raw', 'rmssd_raw', 'lf_hf_raw', 'br_raw',

    # 2. 개인 기준값 (_base)
    #    → "이 사람의 평소 상태가 어떤 사람인가"
    'hr_base', 'sdnn_base', 'rmssd_base', 'lf_hf_base', 'br_base',

    # 3. 기준 대비 z-score (_znorm)
    #    → "이 사람 기준으로 지금 얼마나 비정상인가"
    'hr_znorm', 'sdnn_znorm', 'rmssd_znorm', 'lf_hf_znorm', 'br_znorm',

    # 4. 신호 품질
    #    → 낮으면 위 값들 신뢰도 낮음을 모델에게 알려줌
    'sqi_raw',
]

# ❌ 제외하는 것들
# pnn50  → rmssd와 중복 (상관관계 0.9 이상)
# _diff  → raw랑 base 있으면 중복
# _status → 문자열이라 모델 입력 불가 (해석용으로만 사용)

[상황 예시]
사람 A (평소 HR=55, 저심박자):  현재 HR=80
사람 B (평소 HR=80, 고심박자):  현재 HR=80

  hr_raw  = 80      80     → 둘 다 같음, raw만으론 구분 불가
  hr_base = 55      80     → "A는 원래 낮은 사람"이라는 정보 추가
  hr_znorm= +12.5   0.0    → A는 극도로 비정상, B는 정상

세 가지가 함께 있어야 모델이 정확히 판단 가능

In [5]:
#pip install open-rppg

import rppg
import time
import cv2
import numpy as np
import pandas as pd
from datetime import datetime
import os

# ── 설정 ───────────────────────────────────────────────
SQI_THRESHOLD   = 0.6
WINDOW_SEC      = 20
SAMPLE_INTERVAL = 2
BASELINE_N      = 15
OUTPUT_CSV      = f"rppg_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# ───────────────────────────────────────────────────────

FEATURES = ['hr', 'sqi', 'sdnn', 'rmssd', 'pnn50', 'lf_hf', 'br']

AI_FEATURES = [
    'hr_raw',    'sdnn_raw',    'rmssd_raw',   'lf_hf_raw',   'br_raw',
    'hr_base',   'sdnn_base',   'rmssd_base',  'lf_hf_base',  'br_base',
    'hr_znorm',  'sdnn_znorm',  'rmssd_znorm', 'lf_hf_znorm', 'br_znorm',
    'sqi_raw',
]

raw_buffer = []
clean_rows = []
baseline   = {}

model = rppg.Model('ME-flow.rlap')


def append_csv(normed, path):
    """
    정규화된 샘플 1개를 CSV에 즉시 한 줄 추가.
    파일이 없으면 헤더 포함해서 새로 생성.
    """
    cols   = ['timestamp'] + AI_FEATURES
    exists = os.path.exists(path)

    # AI_FEATURES 중 실제로 normed에 있는 것만 추출
    row_data = {c: normed.get(c, float('nan')) for c in cols}
    df_row   = pd.DataFrame([row_data])

    # 파일 없으면 헤더 포함, 있으면 헤더 없이 append
    df_row.to_csv(path, mode='a', header=not exists, index=False)


def extract_row(result, ts):
    try:
        hrv = result.get('hrv') or {}

        br_raw = hrv.get('breathingrate', float('nan'))
        br = br_raw * 60 if (not np.isnan(br_raw) and br_raw < 2.0) else br_raw

        pnn50_raw = hrv.get('pnn50', float('nan'))
        pnn50 = pnn50_raw * 100 if (not np.isnan(pnn50_raw) and pnn50_raw <= 1.0) else pnn50_raw

        row = {
            'timestamp': ts,
            'hr'       : result['hr'],
            'sqi'      : result.get('SQI', float('nan')),
            'sdnn'     : hrv.get('sdnn',  float('nan')),
            'rmssd'    : hrv.get('rmssd', float('nan')),
            'pnn50'    : pnn50,
            'lf_hf'    : hrv.get('LF/HF', float('nan')),
            'br'       : br,
        }
        if row['hr'] is None or np.isnan(row['hr']):
            return None
        return row
    except (KeyError, TypeError):
        return None


def sqi_filter(row):
    return not np.isnan(row['sqi']) and row['sqi'] >= SQI_THRESHOLD


def range_filter(row):
    checks = [
        (row['hr'],    30,  200),
        (row['sdnn'],   0,  300),
        (row['rmssd'],  0,  300),
        (row['pnn50'],  0,  100),
        (row['lf_hf'],  0,   20),
        (row['br'],     4,   60),
    ]
    for val, lo, hi in checks:
        if not np.isnan(val) and not (lo <= val <= hi):
            print(f"[SKIP] 생리 범위 이탈 — "
                  f"HR={row['hr']:.1f}, SDNN={row['sdnn']:.1f}, "
                  f"RMSSD={row['rmssd']:.1f}, LF/HF={row['lf_hf']:.2f}, "
                  f"BR={row['br']:.1f}, PNN50={row['pnn50']:.1f}")
            return False
    return True


def compute_baseline(rows):
    df = pd.DataFrame(rows)[FEATURES]
    return {
        feat: {'mean': df[feat].mean(), 'std': df[feat].std()}
        for feat in FEATURES
    }


def interpret_znorm(znorm):
    if np.isnan(znorm):   return 'unknown'
    if   znorm >  2.0:    return 'very_high'
    elif znorm >  1.5:    return 'high'
    elif znorm < -2.0:    return 'very_low'
    elif znorm < -1.5:    return 'low'
    else:                 return 'normal'


def normalize(row, base):
    normed = {'timestamp': row['timestamp']}
    for feat in FEATURES:
        mean  = base[feat]['mean']
        std   = base[feat]['std']
        val   = row[feat]
        znorm = (val - mean) / std if std > 0 else 0.0

        normed[f'{feat}_raw']    = round(val,        4)
        normed[f'{feat}_base']   = round(mean,        4)
        normed[f'{feat}_diff']   = round(val - mean,  4)
        normed[f'{feat}_znorm']  = round(znorm,       4)
        normed[f'{feat}_status'] = interpret_znorm(znorm)
    return normed


# ── 메인 루프 ───────────────────────────────────────────
print(f"rPPG 데이터 수집 시작. 저장 경로: {OUTPUT_CSV}")
print("'q'로 종료\n")

with model.video_capture(0):
    last_sample_time = 0
    baseline_done    = False
    saved_count      = 0     # 실시간 저장된 행 수

    for frame, box in model.preview:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        now   = time.time()

        if now - last_sample_time >= SAMPLE_INTERVAL:
            result = model.hr(start=-WINDOW_SEC, return_hrv=True)
            if result:
                row = extract_row(result, ts=now)
                if row:
                    raw_buffer.append(row)

                    # ❶ SQI 필터
                    if not sqi_filter(row):
                        print(f"[SKIP] SQI={row['sqi']:.2f} — 신호 품질 미달")
                        last_sample_time = now
                        continue

                    # ❷ 범위 필터
                    if not range_filter(row):
                        last_sample_time = now
                        continue

                    # ❸ 기준선 수집
                    if not baseline_done:
                        clean_rows.append(row)
                        print(f"[BASELINE {len(clean_rows)}/{BASELINE_N}] "
                              f"HR={row['hr']:.1f}")
                        if len(clean_rows) >= BASELINE_N:
                            baseline = compute_baseline(clean_rows)
                            baseline_done = True
                            print("✅ 기준선 확정:")
                            for k, v in baseline.items():
                                print(f"   {k:6s}: 평균={v['mean']:.2f}, "
                                      f"std={v['std']:.2f}")

                    # ❹ 정규화 → 즉시 CSV 한 줄 추가
                    else:
                        normed = normalize(row, baseline)
                        clean_rows.append(normed)

                        # ✅ 측정 즉시 CSV에 한 줄 append
                        append_csv(normed, OUTPUT_CSV)
                        saved_count += 1

                        # 콘솔 출력
                        print("\n" + "─" * 60)
                        print(f"[CLEAN #{saved_count}] "
                              f"{datetime.now().strftime('%H:%M:%S')} "
                              f"→ {OUTPUT_CSV} 저장됨")
                        print(f"  {'피처':6s}  {'현재':>7}  {'기준':>7}  "
                              f"{'차이':>7}  {'z':>6}  상태")
                        for feat in FEATURES:
                            print(
                                f"  {feat:6s}  "
                                f"{normed[f'{feat}_raw']:>7.2f}  "
                                f"{normed[f'{feat}_base']:>7.2f}  "
                                f"{normed[f'{feat}_diff']:>+7.2f}  "
                                f"{normed[f'{feat}_znorm']:>+6.2f}  "
                                f"{normed[f'{feat}_status']}"
                            )
                        print("─" * 60)

            last_sample_time = now

        # ── HUD ─────────────────────────────────────────
        status = (f"Saved: {saved_count} rows | "
                  f"Baseline: {'OK' if baseline_done else 'Collecting...'}")
        cv2.putText(frame, status, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 128), 2)

        if box is not None:
            y1, y2 = box[0]; x1, x2 = box[1]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.imshow("rPPG Collector", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# ── 종료 요약 ────────────────────────────────────────────
print(f"\n✅ 수집 완료: 총 {saved_count}개 샘플 저장 → {OUTPUT_CSV}")

if saved_count > 0:
    df_final = pd.read_csv(OUTPUT_CSV)
    print(f"📊 컬럼: {list(df_final.columns)}")
    print(df_final.describe().round(3))
else:
    pd.DataFrame(raw_buffer).to_csv("rppg_raw_fallback.csv", index=False)
    print("⚠️  정제 샘플 없음 — raw 데이터만 저장됨")

rPPG 데이터 수집 시작. 저장 경로: rppg_clean_20260405_103242.csv
'q'로 종료

[SKIP] SQI=0.49 — 신호 품질 미달
[BASELINE 1/15] HR=84.2
[BASELINE 2/15] HR=81.0
[BASELINE 3/15] HR=80.5
[BASELINE 4/15] HR=80.5
[BASELINE 5/15] HR=80.1
[BASELINE 6/15] HR=80.1
[BASELINE 7/15] HR=81.0
[BASELINE 8/15] HR=81.0
[BASELINE 9/15] HR=80.0
[BASELINE 10/15] HR=79.9
[BASELINE 11/15] HR=80.7
[BASELINE 12/15] HR=79.7
[BASELINE 13/15] HR=72.3
[SKIP] 생리 범위 이탈 — HR=74.3, SDNN=59.9, RMSSD=53.0, LF/HF=4.89, BR=0.0, PNN50=33.3
[BASELINE 14/15] HR=73.7
[BASELINE 15/15] HR=73.8
✅ 기준선 확정:
   hr    : 평균=79.22, std=3.27
   sqi   : 평균=0.76, std=0.05
   sdnn  : 평균=55.54, std=12.15
   rmssd : 평균=63.37, std=9.70
   pnn50 : 평균=57.90, std=18.03
   lf_hf : 평균=2.14, std=1.95
   br    : 평균=10.48, std=3.99

────────────────────────────────────────────────────────────
[CLEAN #1] 10:33:21 → rppg_clean_20260405_103242.csv 저장됨
  피처           현재       기준       차이       z  상태
  hr        74.61    79.22    -4.61   -1.41  normal
  sqi        0.83     0.

Exception in thread Thread-22:
Traceback (most recent call last):
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 950, in _bootstrap_inner
    self.run()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\threading.py", line 888, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 723, in <lambda>
    self.run = threading.Thread(target=lambda:self.__process_video_capture(vid_path, api))
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 841, in __process_video_capture
    self.update_frame(img, ts)
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 535, in __exit__
    self.wait_completion()
  File "c:\Users\jisci\miniconda3\envs\ai\lib\site-packages\rppg\main.py", line 737, in wait_completion
    

PermissionError: [Errno 13] Permission denied: 'rppg_clean_20260405_103242.csv'